# Кибериммунный подход к разработке. Учебный пример "Светофор"

## Об авторе 

Этот блокнот разработан для вас Сергеем Соболевым, sergey.p.sobolev@kaspersky.com

Больше информации о кибериммунном подходе можно найти на странице https://github.com/sergey-sobolev/cyberimmune-systems/wiki/%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82

Подписывайтесь на телеграм-канал @learning_cyberimmunity (https://t.me/learning_cyberimmunity)

Обучающие видео на тему кибериммунного подхода вы можете найти на youtube канале https://www.youtube.com/@learning_cyberimmunity/

## О примере 

Светофор - это, на первый взгляд, очень простая система, но она оказывает критическое влияние на безопасность дорожного движения. 
Применим идеи конструктивной безопасности к архитектуре и реализации прототипа светофора. За отправную точку возьмём [код базового примера](https://colab.research.google.com/github/sergey-sobolev/cyberimmune-systems-basic-demo-notebook01/blob/main/cyberimmunity-basics.ipynb) и немного его переработаем.

Для переработки будем использовать описание архитектуры решения, которое обсуждалось на занятии.

## Простая реализация выбранной политики архитектуры

Будем использовать политику архитектуры, показанную на рис. 1.

![Рис. 1. Политика архитектуры](images/tl-archpol-0.02.png)

Рис. 1. Политика архитектуры светофора

1. Создадим функциональные компоненты (сущности 1-4) и монитор безопасности, который будет контролировать их взаимодействие, в том числе реализовывать контроль конфигураций светофора (сущность №5 на архитектурной диаграмме)
2. Определим политики безопасности
3. Сымитируем запрос на изменение режима для проверки работы всех элементов

- В качестве интерфейса взаимодействия используем очереди сообщений, у каждой сущности есть своя «персональная» очередь, ассоциированная с ней
- Компоненты 1-4 отправляют сообщения только в очередь monitor сущности SecurityMonitor
- SecurityMonitor проверяет сообщения на соответствие политикам безопасности, в случае положительного решения перенаправляет сообщение в очередь соответствующей сущности

В коде назовём сущности следующим образом
1. Связь - CitySystemConnector
2. Система управления светофора - ControlSystem
3. Управление светодиодами - LightsGPIO
4. Система диагностики - SelfDiagnosticsSystem

Логику контроля режимов светофора (компонент №5 на рис. 1) реализуем в виде политик безопасности в мониторе безопасности

![Рис. 2. Политика архитектуры с именами классов](images/tl-archpol-code.png)

Рис. 2. Политика архитектуры с именами классов

Очередь событий для монитора безопасности: все запросы от сущностей друг к другу должны отправляться только в неё

In [9]:
from multiprocessing import Queue
monitor_events_queue = Queue()

Зафиксируем формат сообщений

In [10]:
from dataclasses import dataclass


@dataclass
class Event:
    source: str       # отправитель
    destination: str  # получатель
    operation: str    # чего хочет (запрашиваемое действие)
    parameters: str   # с какими параметрами

### Монитор безопасности

Ниже в методе _check_policies можно увидеть пример политики безопасности:

```python
if event.source == "ControlSystem" \
        and event.destination == "LightsGPIO" \
        and event.operation == "set_mode" \ 
        and self._check_mode(event.operation):
    authorized = True
```            

в этом примере проверяется отправитель сообщения, получатель, запрашиваемая операция и даже параметры операции. Это максимально жёсткий вариант, очевидно, в зависимости от ситуации количество проверок можно уменьшить.

А пока это место для экспериментов, как можно из монитора безопасности заблокировать взаимодействие между сущностями.

In [11]:
from multiprocessing import Queue, Process
from multiprocessing.queues import Empty
from multiprocessing import Event as MpEvent
from dataclasses import dataclass
from time import sleep
import json
import time 

class Event:
    def __init__(self, source, destination, operation, parameters):
        self.source = source
        self.destination = destination
        self.operation = operation
        self.parameters = parameters
    def __repr__(self):
        return f"Event({self.source}→{self.destination}, {self.operation})"

@dataclass
class ControlEvent:
    operation: str

traffic_lights_allowed_configurations = [
    # Новичок - мигающий желтый
    {"direction_1": "red", "direction_2": "green"},
    {"direction_1": "red", "direction_2": "red"},
    {"direction_1": "red", "direction_2": "yellow"},
    {"direction_1": "yellow", "direction_2": "yellow"},
    {"direction_1": "off", "direction_2": "off"},
    {"direction_1": "green", "direction_2": "red"},
    {"direction_1": "green", "direction_2": "yellow"},
    {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking"},

    # Средний уровень 
    {"direction_1": "green", "direction_2": "red", "turn_right": True},
    {"direction_1": "red", "direction_2": "green", "turn_right": True},
    {"direction_1": "green_arrow", "direction_2": "red"},
    {"direction_1": "red", "direction_2": "green_arrow"},
    {"direction_1": "red", "direction_2": "red", "right_arrow_1": True},
    {"direction_1": "red", "direction_2": "red", "right_arrow_2": True},
    {"direction_1": "red", "direction_2": "green", "right_arrow_1": True},
    {"direction_1": "green", "direction_2": "red", "right_arrow_2": True},
    {"direction_1": "red", "direction_2": "red", "left_arrow_1": True},
    {"direction_1": "red", "direction_2": "red", "left_arrow_2": True},
    {"direction_1": "red", "direction_2": "green", "left_arrow_1": True},
    {"direction_1": "green", "direction_2": "red", "left_arrow_2": True},
]


class Monitor(Process):
    def __init__(self, events_q: Queue):   
        super().__init__()
        self._events_q = events_q
        self._control_q = Queue()
        self._entity_queues = {}
        self._force_quit = False

    def add_entity_queue(self, entity_id: str, queue: Queue):
        print(f"[монитор] регистрируем сущность {entity_id}")
        self._entity_queues[entity_id] = queue

    def _check_mode(self, mode_str: str) -> bool:
        try:
            mode = json.loads(mode_str)
            print(f"[монитор] проверяем конфигурацию {mode}")
            return mode in traffic_lights_allowed_configurations
        except:
            return False

    def _check_policies(self, event):
        print(f'[монитор] обрабатываем событие {event}')
        if not isinstance(event, Event):
            return False

        # 1. Установка режима светофора 
        if (event.source == "ControlSystem" and event.destination == "LightsGPIO"
                and event.operation == "set_mode" and self._check_mode(event.parameters)):
            return True

        # 2. Отправка статуса от LightsGPIO в SelfDiagnostics 
        if (event.source == "LightsGPIO" and event.destination == "SelfDiagnostics"
                and event.operation == "status"):
            return True

        if (event.source == "ControlSystem" and event.destination == "LightsGPIO"
                and event.operation in ["enable_turn_right", "enable_turn_left"]):
            try:
                params = json.loads(event.parameters)
                if params.get("direction") in ["direction_1", "direction_2"]:
                    return True
            except:
                pass

        # 4. Диагностические взаимодействия 
        if (event.source == "LightsGPIO" and event.destination == "SelfDiagnostics"
                and event.operation in ["status_update", "error_report", "health_check"]):
            return True

        if (event.source == "ControlSystem" and event.destination == "SelfDiagnostics"
                and event.operation == "request_diagnostics"):
            return True

        if (event.source == "SelfDiagnostics" and event.destination in ["ControlSystem", "LightsGPIO"]
                and event.operation in ["diagnostics_report", "system_alert", "maintenance_command"]):
            return True

        if event.operation == "emergency_stop" and event.destination == "ControlSystem":
            return True

        # Продвинутый уровень: взаимодействие с CitySystemConnector
        if (event.source == "CitySystemConnector" and event.destination == "ControlSystem"
                and event.operation in ["set_green_duration", "set_mode", "get_status"]):
            return True

        if (event.source == "ControlSystem" and event.destination == "CitySystemConnector"
                and event.operation == "status_report"):
            return True

        if (event.source == "SelfDiagnostics" and event.destination == "CitySystemConnector"
                and event.operation == "diagnostics_report"):
            return True

        print("[монитор] событие не разрешено политиками безопасности")
        return False

    def _proceed(self, event):
        print(f'[монитор] отправляем запрос {event}')
        try:
            dst_q = self._entity_queues[event.destination]
            dst_q.put(event)
        except Exception as e:
            print(f"[монитор] ошибка выполнения запроса {e}")

    def run(self):
        print('[монитор] старт')
        while not self._force_quit:
            try:
                event = self._events_q.get_nowait()
                if self._check_policies(event):
                    self._proceed(event)
            except Empty:
                sleep(0.2)
            except Exception as e:
                print(f"[монитор] ошибка обработки: {e}")
            self._check_control_q()
        print('[монитор] завершение работы')

    def stop(self):
        self._control_q.put(ControlEvent('stop'))

    def _check_control_q(self):
        try:
            req = self._control_q.get_nowait()
            if isinstance(req, ControlEvent) and req.operation == 'stop':
                self._force_quit = True
        except Empty:
            pass

### Сущность ControlSystem

Эта сущность отправляет сообщение для другой сущности (LightsGPIO)

In [12]:
class ControlSystem(Process):
    def __init__(self, monitor_queue: Queue, shutdown_event: MpEvent):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.shutdown_event = shutdown_event
        self.green_duration = 5  # секунд
        self.current_phase = 0
        self.modes = [
            {"direction_1": "green", "direction_2": "red"},
            {"direction_1": "red", "direction_2": "green"}
        ]

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        
        # --- Для демонстрации уровня "Новичок" отправляем недопустимый режим ---
        invalid_mode = {"direction_1": "green", "direction_2": "green"}
        event_invalid = Event(source=self.__class__.__name__,
                              destination='LightsGPIO',
                              operation='set_mode',
                              parameters=json.dumps(invalid_mode))
        self.monitor_queue.put(event_invalid)

        last_switch = time.time()

        while not self.shutdown_event.is_set():
            # Переключение фазы по таймеру
            if time.time() - last_switch >= self.green_duration:
                self.current_phase = (self.current_phase + 1) % len(self.modes)
                mode = self.modes[self.current_phase]
                event = Event(source=self.__class__.__name__,
                              destination='LightsGPIO',
                              operation='set_mode',
                              parameters=json.dumps(mode))
                self.monitor_queue.put(event)
                last_switch = time.time()
                print(f'[{self.__class__.__name__}] переключение на режим {mode}')

            # Обработка входящих команд (от CitySystemConnector)
            try:
                cmd = self._own_queue.get_nowait()
                if cmd.operation == 'set_green_duration':
                    params = json.loads(cmd.parameters)
                    self.green_duration = params['duration']
                    print(f'[{self.__class__.__name__}] длительность изменена на {self.green_duration}')
                elif cmd.operation == 'set_mode':
                    mode = json.loads(cmd.parameters)
                    event = Event(source=self.__class__.__name__,
                                  destination='LightsGPIO',
                                  operation='set_mode',
                                  parameters=json.dumps(mode))
                    self.monitor_queue.put(event)
                elif cmd.operation == 'get_status':
                    status = {
                        'current_mode': self.modes[self.current_phase],
                        'green_duration': self.green_duration
                    }
                    response = Event(source=self.__class__.__name__,
                                     destination='CitySystemConnector',
                                     operation='status_report',
                                     parameters=json.dumps(status))
                    self.monitor_queue.put(response)
            except Empty:
                pass
            sleep(0.1)

        print(f'[{self.__class__.__name__}] завершение работы')


        
        
        
        
class CitySystemConnector(Process):
    def __init__(self, monitor_queue: Queue, shutdown_event: MpEvent):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.shutdown_event = shutdown_event

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        start_time = time.time()
        while not self.shutdown_event.is_set():
            # Раз в 15 секунд отправляем запрос статуса
            if time.time() - start_time > 15:
                req = Event(source=self.__class__.__name__,
                            destination='ControlSystem',
                            operation='get_status',
                            parameters='{}')
                self.monitor_queue.put(req)
                start_time = time.time()

            # Обработка входящих сообщений (статусы, диагностика)
            try:
                msg = self._own_queue.get_nowait()
                if msg.operation == 'status_report':
                    status = json.loads(msg.parameters)
                    print(f'[{self.__class__.__name__}] получен статус светофора: {status}')
                elif msg.operation == 'diagnostics_report':
                    diag = json.loads(msg.parameters)
                    print(f'[{self.__class__.__name__}] получен диагностический отчёт: {diag}')
                else:
                    print(f'[{self.__class__.__name__}] получено сообщение: {msg}')
            except Empty:
                pass
            sleep(0.5)

        print(f'[{self.__class__.__name__}] завершение работы')


        
        
        
        
        
        
class SelfDiagnostics(Process):
    def __init__(self, monitor_queue: Queue, shutdown_event: MpEvent):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.shutdown_event = shutdown_event

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        start_time = time.time()
        last_report = time.time()
        while not self.shutdown_event.is_set():
            # Раз в 20 секунд отправлять диагностический отчёт в CitySystemConnector
            if time.time() - last_report > 20:
                report = {
                    'status': 'OK',
                    'errors': [],
                    'uptime': time.time() - start_time
                }
                event = Event(source=self.__class__.__name__,
                              destination='CitySystemConnector',
                              operation='diagnostics_report',
                              parameters=json.dumps(report))
                self.monitor_queue.put(event)
                last_report = time.time()

            try:
                event = self._own_queue.get_nowait()
                if event.operation == "request_diagnostics":
                    report = {'status': 'OK'}
                    response = Event(source=self.__class__.__name__,
                                     destination=event.source,
                                     operation='diagnostics_report',
                                     parameters=json.dumps(report))
                    self.monitor_queue.put(response)
                elif event.operation == "status":
                    print(f"[{self.__class__.__name__}] получен статус: {event.parameters}")
            except Empty:
                pass
            sleep(0.5)

        print(f'[{self.__class__.__name__}] завершение работы')
        


### Сущность LightsGPIO

Эта сущность ждёт входящее сообщение в течение заданного периода времени, если получает - обрабатывает и завершает работу или выходит по таймауту.

In [13]:
from multiprocessing import Queue, Process
from time import sleep

class LightsGPIO(Process):
    def __init__(self, monitor_queue: Queue, shutdown_event: MpEvent):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.shutdown_event = shutdown_event

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        while not self.shutdown_event.is_set():
            try:
                event = self._own_queue.get_nowait()
                if event.operation == "set_mode":
                    print(f"[{self.__class__.__name__}] получен режим: {event.parameters}")
                    # Отправляем статус в самодиагностику
                    status_event = Event(
                        source="LightsGPIO",
                        destination="SelfDiagnostics",
                        operation="status",
                        parameters="OK"
                    )
                    self.monitor_queue.put(status_event)
            except Empty:
                sleep(0.2)
        print(f'[{self.__class__.__name__}] завершение работы')

In [16]:
from multiprocessing import Event as MpEvent

shutdown_event = MpEvent()

# Создаём монитор и все сущности
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue, shutdown_event)
lights_gpio = LightsGPIO(monitor_events_queue, shutdown_event)
self_diagnostics = SelfDiagnostics(monitor_events_queue, shutdown_event)
city_connector = CitySystemConnector(monitor_events_queue, shutdown_event)

# Регистрируем очереди в мониторе (имена должны совпадать с event.destination)
monitor.add_entity_queue("ControlSystem", control_system.entity_queue())
monitor.add_entity_queue("LightsGPIO", lights_gpio.entity_queue())
monitor.add_entity_queue("SelfDiagnostics", self_diagnostics.entity_queue())
monitor.add_entity_queue("CitySystemConnector", city_connector.entity_queue())

# Запускаем процессы
monitor.start()
control_system.start()
lights_gpio.start()
self_diagnostics.start()
city_connector.start()

# Даём системе поработать (например, 60 секунд)
try:
    sleep(60)
except KeyboardInterrupt:
    pass
finally:
    # Останавливаем все процессы
    shutdown_event.set()
    monitor.stop()  # для монитора свой механизм остановки
    for p in [control_system, lights_gpio, self_diagnostics, city_connector, monitor]:
        p.join()
    print("Все процессы завершены")

[монитор] регистрируем сущность ControlSystem
[монитор] регистрируем сущность LightsGPIO
[монитор] регистрируем сущность SelfDiagnostics
[монитор] регистрируем сущность CitySystemConnector
[монитор] старт
[ControlSystem] старт
[LightsGPIO] старт
[SelfDiagnostics] старт
[CitySystemConnector] старт
[монитор] обрабатываем событие Event(ControlSystem→LightsGPIO, set_mode)
[монитор] проверяем конфигурацию {'direction_1': 'green', 'direction_2': 'green'}
[монитор] событие не разрешено политиками безопасности
[ControlSystem] переключение на режим {'direction_1': 'red', 'direction_2': 'green'}
[монитор] обрабатываем событие Event(ControlSystem→LightsGPIO, set_mode)
[монитор] проверяем конфигурацию {'direction_1': 'red', 'direction_2': 'green'}
[монитор] отправляем запрос Event(ControlSystem→LightsGPIO, set_mode)
[LightsGPIO] получен режим: {"direction_1": "red", "direction_2": "green"}
[монитор] обрабатываем событие Event(LightsGPIO→SelfDiagnostics, status)
[монитор] отправляем запрос Event(Li

### Инициализируем монитор и сущности

In [17]:
monitor = Monitor(monitor_events_queue)
control_system = ControlSystem(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue)

TypeError: __init__() missing 1 required positional argument: 'shutdown_event'

регистрируем очереди сущностей в мониторе

In [ ]:
monitor.add_entity_queue(control_system.__class__.__name__, control_system.entity_queue())
monitor.add_entity_queue(lights_gpio.__class__.__name__, lights_gpio.entity_queue())

### Запускаем всё

Ожидаемая последовательность событий

![Диаграмма последовательности вызовов](https://www.plantuml.com/plantuml/png/dPBVIiCm6CNlynIvdtk1NSZ02n4KXJr1w886-cUqcR0xgoBpIdoJLgqhtRg-mfStygJPM3js9OLyoPTpVia97ITQn7eU-6o6gZmr4w7c5r6euyYVB18j0ouIxhb6JpIHtZnMUd4JXKf7iPK5RjgJNQlx1vrStbtTMeNVhXZR0VdmV6yQ34QSQjhI5sNcWrE5QMrUgQHlysAUq7oZqcxyq1hbW6KxG8S5KWEBPHMe5MLeOBa6uHd4YWbVSs1JQg1OKktEF3ATSTI2Vk7Os6b6AupGerctn3uM5WWfn_OAxGRGr4OogTtktjCzmt3utyZ2q-fHQBb_JrSEv177oHigRB1E2ChOL1vxfPz8ZWjdBhv9ELn5Bw_vfFf4L2fFlZsEtkBBpNjxWH8mkyHWbbZbC1TCXbCsne1Vxmy0)

In [ ]:
monitor.start()
control_system.start()
lights_gpio.start()
sleep(2)

### Теперь останавливаем

In [ ]:
monitor.stop()
control_system.join()
lights_gpio.join()
monitor.join()

## Заключение

В этом блокноте продемонстрирован базовый функционал контролируемого изменения режима работы светофора. 

В примере не реализованы некоторые сущности и большая часть логики работы светофора, которую можно предположить по архитектурной диаграмме. Попробуйте сделать это самостоятельно!

## Упражнения

Уровень "Новичок"

- в коде ControlSystem измените режим на недопустимый (два зелёных) и выполните все ячейки. Убедитесь, что монитор безопасности заблокировал сообщение, как нарушающее политику безопасности
- измените политики безопасности так, чтобы был возможен режим "моргающий жёлтый" (yellow_blinking), переводящий перекрёсток в режим нерегулируемого

Уровень "Средней сложности"

- добавьте политики безопасности для доп. секций со стрелками (поворот налево или направо)
- измените код сущностей, чтобы они не завершали работу после одного сообщения, а работали произвольное время (см. реализацию монитора безопасности)
- реализуйте сущность само-диагностики (SelfDiagnostics) и отправку сообщений от LightsGPIO (необходимо доработать политики безопасности!)

Уровень "Продвинутый"

- измените код сущности ControlSystem, реализуйте смену режимов по таймеру (заданную длительность зелёного по каждому направлению)
- реализуйте сущность CitySystemConnector, которая будет имитировать получение изменения режима, реализуйте взаимодействие CitySystemConnector и ControlSystem (понадобится доработать политики безопасности). Например, изменение длительности зелёного по направлениям или отключение светофора (перевод перекрёстка в режим нерегулируемого)
- реализуйте передачу в компонент CitySystemConnector информации об исправности светофора (статус самодиагностики; текущий режим работы). В компоненте реализуйте вывод в виде сообщений о состоянии системы